# Tugas Kelompok 2: Data Collection Pipeline

## Kelompok sowang

1. Fransisca Ellya Bunaren (2306152286)
2. Micheline Wijaya Limbergh (2306207013)
3. Nisrina Annaisha Sarnadi (2306275960)
4. Rizqya Az Zahra Putri (2306244936)
5. Muhammad Satrio Haryo Kusumo (2006597771)

# Tahap Filtering dan Labeling Relevansi

Notebook ini melanjutkan hasil pembersihan data dari Notebook 1 (`data/interim/cleaned.csv`). Tujuannya adalah menyaring postingan yang berpotensi tidak relevan, mengevaluasi hasil filtering melalui pemeriksaan manual (*spot-check*), serta melakukan koreksi berdasarkan kesepakatan anotator.

**Output:** `relevant.csv`, `removed_relevance.csv`, `spot_check.xlsx`, `labeling_report.json`, dan `relevant_final.csv`, semuanya disimpan dalam `data/labeling/`.

## 0. Persiapan dan Konfigurasi

In [8]:
from pathlib import Path
from datetime import datetime, timezone
import json, re
import pandas as pd
from IPython.display import display

CUR = Path.cwd().resolve()
ROOT = next((q for q in [CUR, CUR.parent] if (q / 'data' / 'interim' / 'cleaned.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Tidak menemukan data/interim/cleaned.csv; buka notebook di root sowang/notebooks atau jalankan Notebook 1.')
INPUT = ROOT / 'data' / 'interim' / 'cleaned.csv'
OUT = ROOT / 'data' / 'labeling'
OUT.mkdir(parents=True, exist_ok=True)

SEED = 2026
WEAK_MIN_HITS = 2
DROP_NON_LITERAL = False
SPOT_REMOVED_N = 40
SPOT_KEPT_N = 100
LABELS = {'relevant', 'irrelevant', 'uncertain'}

print('Input:', INPUT)
print('Output:', OUT)

Input: C:\Users\nisri\OneDrive\Documents\University\Anmedsos\sowang\data\interim\cleaned.csv
Output: C:\Users\nisri\OneDrive\Documents\University\Anmedsos\sowang\data\labeling


## 1. Memuat Dataset

Memastikan kolom penting tersedia dan ID postingan dapat digunakan.

In [9]:
df = pd.read_csv(INPUT, dtype=str, keep_default_na=False, encoding='utf-8-sig')
required = ['id', 'text', 'group', 'keyword', 'created_at', 'source_month']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Kolom wajib tidak ada: {missing}')
if df['id'].eq('').any() or df['id'].duplicated().any():
    raise ValueError('ID kosong/duplikat di cleaned.csv. Periksa Notebook 1.')
dates = pd.to_datetime(df['created_at'], errors='coerce', utc=True)
if dates.isna().any():
    raise ValueError(f'Terdapat {dates.isna().sum()} tanggal tidak valid.')
report1 = ROOT / 'data' / 'interim' / 'validation_report.json'
if report1.exists():
    expected = json.loads(report1.read_text(encoding='utf-8')).get('cleaning', {}).get('after')
    if expected is not None and len(df) != expected:
        raise ValueError(f'Jumlah input {len(df)} != laporan Notebook 1 ({expected}).')
df['is_non_literal'] = df['keyword'].eq('(match_non_literal)')
print('Jumlah data:',len(df))
display(df.groupby(['source_month','group']).size().rename('jumlah').reset_index().head(20))

Jumlah data: 8367


,source_month,group,jumlah
0,2025-10,gentle,83
1,2025-10,lain,1
2,2025-10,otoriter,35
3,2025-10,permisif,36
4,2025-10,umum,545
5,2025-11,gentle,74
6,2025-11,otoriter,28
7,2025-11,permisif,9
8,2025-11,umum,586
9,2025-12,demokratis,1


## 2. Filtering Relevansi Otomatis

Mengidentifikasi postingan yang diduga promosi, spam, atau di luar pembahasan parenting menggunakan aturan berbasis pola teks (*rule-based filtering*). Pola kuat dan gabungan beberapa pola lemah digunakan untuk menentukan kandidat yang dibuang.

Hasil dibagi menjadi dua kelompok: **kandidat relevan** (lolos aturan) dan **kandidat tidak relevan** (terdeteksi oleh aturan). Keputusan ini belum merupakan label kebenaran (*ground truth*). Hashtag `#fyp`/`#foryou` dan penanda `(match_non_literal)` tidak otomatis menyebabkan post dibuang.

In [10]:
SPAM_STRONG = [
    r'\bjav\b', r'\bbokep\b', r'\bporn\w*', r'\bxxx\b', r'\bvcs\b', r'\bopen\s*bo\b',
    r'\[[a-z]{2,6}-\d{2,5}\]',
    r'\bopen\s*order\b', r'\bopen\s*po\b', r'\bpre\s*-?\s*order\b', r'\bready\s*stock\b',
    r'\b(?:gratis|free)\s*ongkir\b', r'\bcod\b', r'\bwa\.me\b', r'\bklik\s+link\b',
    r'\blink\s+(?:di\s+)?bio\b', r'\bdm\s+(?:untuk|aja|saja|ya)\b', r'\bhubungi\s+(?:kami|admin)\b',
    r'\bdaftar\s+sekarang\b', r'\bkuota\s+terbatas\b', r'\bdiskon\b',
    r'\bshopee\b|\btokopedia\b|\bbit\.ly\b|\blinktr\.ee\b', r'\baffiliate\b',
    r'(?<!\d)(?:\+62|62|0)8\d{8,11}(?!\d)', r'#alternateuniverse',
    r'\bparenting\s*diary\b',
    r'\bcheckout\s+link\b|\bproduct\s+link\b', r'\bdapatkan\s+e-?book'
]
PROMO_WEAK = [
    r'\bwebinar\b', r'\bseminar\b', r'\bworkshop\b', r'\bkelas\b', r'\be-?book\b',
    r'\bdownload\b|\bunduh\b', r'\bgratis\b', r'\bbonus\b', r'\bharga\b', r'\bjual\b|\bdijual\b',
    r'\bkonsultasi\b', r'\bdaftar\b|\bpendaftaran\b', r'\btestimoni\b', r'\brp\.?\s?\d',
    r'\bfollow\b|\bikuti\b', r'\binstagram\b|\bwhatsapp\b',
    r'\bkajian\b', r'\bklik\s+sini\b', r'\bvia\s+@youtube\b', r'\bdonasi\b'
]
assert not any('fyp' in s or 'foryou' in s for s in SPAM_STRONG + PROMO_WEAK)
for pat in SPAM_STRONG + PROMO_WEAK:
    re.compile(pat)

def scan(text, patterns):
    hits = []
    for pat in patterns:
        match = re.search(pat, str(text).casefold())
        if match:
            hits.append(match.group(0))
    return len(hits), ', '.join(dict.fromkeys(hits))

strong = df['text'].map(lambda x: scan(x, SPAM_STRONG))
weak = df['text'].map(lambda x: scan(x, PROMO_WEAK))
df['strong_hits'] = strong.map(lambda x: x[0])
df['strong_terms'] = strong.map(lambda x: x[1])
df['weak_hits'] = weak.map(lambda x: x[0])
df['weak_terms'] = weak.map(lambda x: x[1])

def filter_reason(row):
    if row['strong_hits'] > 0: return 'spam_strong'
    if row['weak_hits'] >= WEAK_MIN_HITS: return 'promo_weak_multi'
    if DROP_NON_LITERAL and row['is_non_literal']: return 'non_literal'
    return ''

df['filter_reason'] = df.apply(filter_reason, axis=1)
df['filter_otomatis'] = df['filter_reason'].map(lambda x: 'dibuang' if x else 'lolos')
removed = df[df['filter_otomatis']=='dibuang'].copy()
kept = df[df['filter_otomatis']=='lolos'].copy()
assert len(removed)+len(kept)==len(df)
print(f'Input: {len(df):,} | kandidat lolos: {len(kept):,} | kandidat dibuang: {len(removed):,}')
display(df['filter_reason'].value_counts().rename_axis('alasan').reset_index(name='jumlah'))
display(pd.crosstab(df['source_month'], df['filter_otomatis']))
if len(removed):
    display(removed[['source_month','group','text','filter_reason','strong_terms','weak_terms']].sample(min(10,len(removed)),random_state=SEED))

Input: 8,367 | kandidat lolos: 8,078 | kandidat dibuang: 289


,alasan,jumlah
0,,8078
1,spam_strong,260
2,promo_weak_multi,29


filter_otomatis,dibuang,lolos
source_month,,
2025-10,21,679
2025-11,49,648
2025-12,31,661
2026-01,12,684
2026-02,7,687
2026-03,7,691
2026-04,10,689
2026-05,46,652
2026-06,42,657


,source_month,group,text,filter_reason,strong_terms,weak_terms
6490,2026-07,umum,eehhh sedih bgt dehh parenting diary udahan 😭😭...,spam_strong,parenting diary,
1021,2025-11,umum,——⚠️JANIN SELALU AKTIF⁉️——⚠️#au #alternateuniv...,spam_strong,#alternateuniverse,
3048,2026-02,umum,Sudah siap tapi kenapa takut banget menikah?\n...,spam_strong,081291509237,"webinar, gratis, daftar"
5757,2026-06,umum,"Member tubatu tuh kenapa vibesnya adem, anak b...",spam_strong,parenting diary,
2408,2026-01,umum,"📒Ibu, Kamu Tidak (Gila) Sendirian \n💰Harga Rp8...",spam_strong,"diskon, shopee","harga, rp8"
5386,2026-05,umum,tak sabar nak tengok eps 5 parenting diary 😭😭😭,spam_strong,parenting diary,
4940,2026-05,umum,ni moa dri kmrn gila gila protes anjing ada aj...,spam_strong,parenting diary,
5913,2026-06,umum,@busanbbunny Bantu jawab. Yoojun anak dari non...,spam_strong,parenting diary,
6736,2026-07,umum,Cimolku sebulan ga ketemu aja udah lupa pernah...,spam_strong,parenting diary,
872,2025-11,umum,Dapatkan ebooknya sekarang di link ini : https...,spam_strong,"product link, dapatkan ebook",


## 3. Manual Labeling

Untuk mengevaluasi filtering, kami melakukan manual labeling dengn cara dua anotator memberikan penilaian berdasarkan **isi dan konteks postingan**, bukan hanya kata kunci atau keputusan otomatis.

| Label | Kriteria |
|---|---|
| `relevant` | Membahas praktik, pengalaman, pendapat, atau dampak pengasuhan anak secara substantif. |
| `irrelevant` | Tidak membahas pengasuhan anak secara substantif. |
| `uncertain` | Informasi atau konteks tidak cukup untuk mengambil keputusan. |

Anotator A dan B  mengisi label **secara independen**. Kategori `uncertain` dipakai untuk kasus yang memang memerlukan pemeriksaan lebih lanjut.

## 4. Membuat Sampel untuk Spot-Check

Mengambil sampel dari postingan yang **lolos** dan **dibuang** oleh filter. Sampel disimpan dalam satu file `spot_check.xlsx` untuk dilakukan labeling manual.

In [11]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
import random

def stratified_sample(frame, size, seed):
    if frame.empty or size <= 0:
        return frame.head(0).copy()
    size = min(size, len(frame))
    strata = list(frame.groupby(['source_month', 'group'], sort=True))
    order = list(range(len(strata)))
    random.Random(seed).shuffle(order)
    idx = [strata[j][1].sample(n=1, random_state=seed+j).index[0] for j in order[:size]]
    if len(idx) < size:
        rest = frame.drop(index=idx)
        idx.extend(rest.sample(n=size-len(idx), random_state=seed+981).index.tolist())
    return frame.loc[idx].copy()

s_removed = stratified_sample(removed, SPOT_REMOVED_N, SEED)
s_kept = stratified_sample(kept, SPOT_KEPT_N, SEED + 1)
sample = pd.concat([s_removed, s_kept], ignore_index=True)
if not sample.empty:
    sample = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)
sample.insert(0, 'item_id', [f'SC{i+1:03d}' for i in range(len(sample))])
cols = ['item_id', 'id', 'source_month', 'group', 'text', 'filter_otomatis', 'filter_reason', 'strong_terms', 'weak_terms', 'keyword']
audit = sample[cols].copy()
audit['label_A'] = ''
audit['label_B'] = ''
audit['catatan'] = ''
spot_path = OUT / 'spot_check.xlsx'

# Jika sudah pernah diisi, jangan hilangkan penilaian. Jika sampel berubah,
# jangan menggunakan penilaian lama untuk sampel baru.
if spot_path.exists():
    old = pd.read_excel(spot_path, dtype=str, keep_default_na=False)
    if not {'item_id', 'id', 'label_A', 'label_B'}.issubset(old.columns):
        raise ValueError('spot_check.xlsx lama tidak memiliki format dua anotator. Pindahkan/hapus file lama lalu jalankan ulang cell ini.')
    if list(old['id']) != list(audit['id']):
        raise ValueError('Sampel berubah tetapi spot_check.xlsx lama masih ada. Pindahkan/hapus file lama, lalu jalankan ulang cell ini.')
    print('spot_check.xlsx sudah ada dan cocok dengan sampel, anotasi lama dipertahankan.')
else:
    # Tulis Excel secara aman (teks X jangan diinterpretasi sebagai rumus Excel).
    audit.to_excel(spot_path, index=False, engine='openpyxl')
    wb = load_workbook(spot_path)
    ws = wb.active
    ws.title = 'spot_check'
    ws.freeze_panes = 'E2'
    for name, width in {'A':11,'B':24,'C':16,'D':18,'E':85,'F':20,'G':22,'H':25,'I':25,'J':25,'K':18,'L':18,'M':40}.items():
        ws.column_dimensions[name].width = width
    for cell in ws[1]:
        cell.font = Font(bold=True)
    dv = DataValidation(type='list', formula1='"relevant,irrelevant,uncertain"', allow_blank=True)
    ws.add_data_validation(dv)
    for letter in ['K','L']:
        dv.add(f'{letter}2:{letter}{len(audit)+1}')
    for row in ws.iter_rows(min_row=2):
        for c in row:
            if isinstance(c.value, str):
                c.data_type = 's'
            c.alignment = Alignment(vertical='top', wrap_text=True)
        for j in [10, 11]:
            row[j].fill = PatternFill('solid', fgColor='FFF2CC')
    wb.save(spot_path)
    print('Dibuat:', spot_path)
print('Sampel audit:', len(audit), '| kandidat dibuang:', len(s_removed), '| kandidat lolos:', len(s_kept))
display(pd.crosstab(sample['source_month'],sample['filter_otomatis']) if len(sample) else pd.DataFrame())

Dibuat: C:\Users\nisri\OneDrive\Documents\University\Anmedsos\sowang\data\labeling\spot_check.xlsx
Sampel audit: 140 | kandidat dibuang: 40 | kandidat lolos: 100


filter_otomatis,dibuang,lolos
source_month,,
2025-10,4,7
2025-11,9,7
2025-12,3,7
2026-01,1,10
2026-02,1,8
2026-03,2,6
2026-04,2,13
2026-05,4,7
2026-06,5,9


## 5. Menyimpan Hasil Filtering

Menyimpan dua dataset hasil penyaringan, yaitu `relevant.csv` untuk postingan yang lolos filter dan `removed_relevance.csv` untuk postingan yang dibuang beserta alasan. Ringkasan jumlah data dan pengaturan proses dicatat dalam `labeling_report.json`.

In [12]:
kept.to_csv(OUT / 'relevant.csv', index=False, encoding='utf-8-sig')
removed.to_csv(OUT / 'removed_relevance.csv', index=False, encoding='utf-8-sig')
report = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'input_rows': int(len(df)),
    'candidate_relevant': int(len(kept)),
    'candidate_removed': int(len(removed)),
    'removed_by_reason': {str(k): int(v) for k, v in removed['filter_reason'].value_counts().items()},
    'by_month': {str(m): {str(k): int(v) for k,v in counts.items()}
                 for m, counts in df.groupby('source_month')['filter_otomatis'].value_counts().unstack(fill_value=0).to_dict(orient='index').items()},
    'sample_rows': int(len(sample)),
    'notes': ['filter regex bersifat kandidat, belum verifikasi penuh', 'non_literal tidak dibuang otomatis', 'sentimen belum dianalisis']
}
report_path = OUT / 'labeling_report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('Output: relevant.csv, removed_relevance.csv, spot_check.xlsx, labeling_report.json')

Output: relevant.csv, removed_relevance.csv, spot_check.xlsx, labeling_report.json


## 6. Evaluasi Manual Labeling

Evaluasi ini menghitung tingkat kesepakatan anotator dan **Cohen's Kappa**, serta memeriksa indikasi postingan relevan yang dibuang dan postingan tidak relevan yang lolos. Hasilnya ditambahkan ke `labeling_report.json` tanpa membuat file output tambahan.

In [15]:
labels = pd.read_excel(spot_path, dtype=str, keep_default_na=False)
if list(labels['id']) != list(audit['id']):
    raise ValueError('spot_check.xlsx tidak sesuai sampel saat ini.')
allowed = {'relevant', 'irrelevant', 'uncertain'}
for c in ['label_A','label_B']:
    labels[c] = labels[c].astype(str).str.strip().str.lower()
    invalid = set(labels[c]) - allowed - {''}
    if invalid:
        raise ValueError(f'Label tidak valid di {c}: {invalid}')
complete = labels[labels['label_A'].isin(allowed) & labels['label_B'].isin(allowed)].copy()
print(f'Anotator A: {labels["label_A"].isin(allowed).sum()}/{len(labels)}')
print(f'Anotator B: {labels["label_B"].isin(allowed).sum()}/{len(labels)}')
if len(complete) == 0:
    print('Belum ada pasangan lengkap. Isi Excel dan jalankan cell evaluasi lagi.')
else:
    agreement = float((complete['label_A'] == complete['label_B']).mean())
    a = complete['label_A'].value_counts(normalize=True).reindex(sorted(allowed),fill_value=0)
    b = complete['label_B'].value_counts(normalize=True).reindex(sorted(allowed),fill_value=0)
    pe = float((a*b).sum())
    kappa = (agreement-pe)/(1-pe) if pe<1 else (1.0 if agreement==1 else float('nan'))
    accepted = complete[(complete['label_A']==complete['label_B']) & (complete['label_A']!='uncertain')]
    wrong_removed = accepted[(accepted['filter_otomatis']=='dibuang') & (accepted['label_A']=='relevant')]
    wrong_kept = accepted[(accepted['filter_otomatis']=='lolos') & (accepted['label_A']=='irrelevant')]
    disputed = complete[(complete['label_A']!=complete['label_B']) | (complete['label_A']=='uncertain')]
    display(pd.crosstab(complete['label_A'], complete['label_B']))
    print(f'Kesepakatan: {agreement:.1%}; Cohen kappa: {kappa:.3f}')
    print(f'Salah dibuang (kasus disepakati): {len(wrong_removed)}')
    print(f'Salah lolos (kasus disepakati): {len(wrong_kept)}')
    print(f'Perbedaan/uncertain: {len(disputed)}')
    if len(wrong_removed): display(wrong_removed[['text','filter_reason','label_A']].head(10))
    if len(wrong_kept): display(wrong_kept[['text','filter_reason','label_A']].head(10))
    if len(disputed): display(disputed[['text','label_A','label_B']].head(10))
    current = json.loads(report_path.read_text(encoding='utf-8'))
    current['audit'] = {
        'pairs_completed': int(len(complete)), 'sample_size': int(len(labels)),
        'raw_agreement': round(agreement,4), 'cohen_kappa': round(float(kappa),4),
        'wrong_removed_agreed': int(len(wrong_removed)), 'wrong_kept_agreed': int(len(wrong_kept)),
        'disagreement_or_uncertain': int(len(disputed)),
        'note':'metrik audit dihitung pada pasangan yang dilabeli; keputusan akhir kasus ambigu perlu diskusi'
    }
    report_path.write_text(json.dumps(current,ensure_ascii=False,indent=2),encoding='utf-8')
    print('Evaluasi ditambahkan ke labeling_report.json')

Anotator A: 110/140
Anotator B: 110/140


label_B,irrelevant,relevant,uncertain
label_A,,,
irrelevant,38,0,0
relevant,0,66,6


Kesepakatan: 94.5%; Cohen kappa: 0.888
Salah dibuang (kasus disepakati): 22
Salah lolos (kasus disepakati): 28
Perbedaan/uncertain: 6


,text,filter_reason,label_A
30,▶ JAV SUB INDO - AKIBAT TERLALU MEMANJAKAN ANA...,spam_strong,relevant
31,Menemani Yoojun melihat pemandangan karena bay...,spam_strong,relevant
35,klo parenting diary tamat kita gak bs lagi don...,spam_strong,relevant
40,Dapatkan ebooknya skrg disini : https://t.co/5...,spam_strong,relevant
51,——⚠️CARA MEMANCING KONTRAKSI⁉️——⚠️#au #alterna...,spam_strong,relevant
54,——⚠️HAL YANG HARUS DIHINDARI DI AWAL KEHAMILAN...,spam_strong,relevant
57,Wipe &amp; Clean Buku Usia Emas Balita Pintar ...,spam_strong,relevant
80,▶ NGOD-165 JAV SUB INDO OB Toge Kesayangan Bos...,spam_strong,relevant
82,gawattt grgr nonton parenting diary kok gue jd...,spam_strong,relevant
85,——⚠️DIABETES SAAT HAMIL⁉️——⚠️#au #alternateuni...,spam_strong,relevant


,text,filter_reason,label_A
32,Gentle parenting?? Hmm sepertinya aku tidak se...,,irrelevant
34,Untuk zaman sekarang gentle parenting tak bole...,,irrelevant
42,@Adventure_36 Inawezekana huyu kalelewa kwa ge...,,irrelevant
43,@simsalabim1419 Strict Parenting sejak dini.,,irrelevant
45,@tttokebi Helicopter parenting _x000D_\nAnak g...,,irrelevant
50,"Sebelum dia ngancem, aku bakal nerapin dari aw...",,irrelevant
60,Kat thread ada dr cadang kurangkan penggunaan ...,,irrelevant
61,@angyjju gue salut sih sm mak bapak jju. finan...,,irrelevant
65,@icukprayogi Helicopter parenting. Kayaknya du...,,irrelevant
70,@jonginsome parenting chanyeol ke sehun,,irrelevant


,text,label_A,label_B
48,@Billy_Crows Parenting yang baik adalah,relevant,uncertain
49,gw literli belakangan ini lg kangen kangenan s...,relevant,uncertain
66,"@haechyn @h Buah kaporit? apa aja yang buah, s...",relevant,uncertain
96,@Jongscong @1 Salam kenal juga sama kamu ya Hu...,relevant,uncertain
126,@tb_siswadi lha iya kalo d sini istilahnya voc...,relevant,uncertain
139,@cogiyeonjun ka hem banyak ketinggalan konten ...,relevant,uncertain


Evaluasi ditambahkan ke labeling_report.json


## 7. Koreksi Manual Hasil Filtering

Koreksi dilakukan berdasarkan kesepakatan kedua anotator:
- **Salah lolos:** Postingan yang lolos filtering otomatis, tetapi dinilai `irrelevant` oleh kedua anotator.
- **Salah dibuang:** Postingan yang dibuang otomatis, tetapi dinilai `relevant` oleh kedua anotator.

Koreksi dilakukan menggunakan ID postingan untuk menghindari penghapusan berdasarkan kata kunci yang terlalu luas.

Hasil koreksi disimpan dalam `relevant_final.csv`, sedangkan `relevant.csv` tetap dipertahankan sebagai hasil filtering awal untuk keperluan audit dan reproducibility.

In [16]:
LABEL_DIR = OUT

# 1. Baca hasil filtering dan spot-check
relevant = pd.read_csv(LABEL_DIR / "relevant.csv", dtype={"id": str})
removed = pd.read_csv(LABEL_DIR / "removed_relevance.csv", dtype={"id": str})
spot = pd.read_excel(LABEL_DIR / "spot_check.xlsx", dtype={"id": str})

# Ambil ID yang benar dari sampel asli
id_reference = audit.set_index("item_id")["id"]

# Pastikan setiap item anotasi terdapat pada sampel asli
if not spot["item_id"].isin(id_reference.index).all():
    raise ValueError(
        "Ada item_id dalam spot_check.xlsx yang tidak ditemukan "
        "pada sampel asli."
    )

# Pulihkan ID berdasarkan item_id, bukan ID yang tersimpan di Excel
spot["id"] = spot["item_id"].map(id_reference)

# Validasi hasil pemetaan
if spot["id"].eq("").any() or spot["id"].isna().any():
    raise ValueError("Ada ID kosong setelah pemetaan ulang.")

if spot["id"].duplicated().any():
    raise ValueError("Ada ID duplikat setelah pemetaan ulang.")

# 3. Ambil postingan yang disepakati salah lolos
salah_lolos = spot[
    (spot["filter_otomatis"] == "lolos") &
    (spot["label_A"] == "irrelevant") &
    (spot["label_B"] == "irrelevant")
]

# 4. Ambil postingan yang disepakati salah dibuang
salah_dibuang = spot[
    (spot["filter_otomatis"] == "dibuang") &
    (spot["label_A"] == "relevant") &
    (spot["label_B"] == "relevant")
]

id_hapus = set(salah_lolos["id"])
id_kembali = set(salah_dibuang["id"])

# 5. Validasi ID
# Pastikan semua ID yang akan dihapus tersedia pada dataset relevant
if not id_hapus.issubset(set(relevant["id"])):
    raise ValueError(
        "Ada ID yang akan dihapus tetapi tidak ditemukan "
        "dalam relevant.csv."
    )

# Pastikan semua ID yang akan dipulihkan tersedia pada dataset removed
if not id_kembali.issubset(set(removed["id"])):
    raise ValueError(
        "Ada ID yang akan dipulihkan tetapi tidak ditemukan "
        "dalam removed_relevance.csv."
    )

# Pastikan tidak ada ID yang muncul pada kedua kelompok koreksi
if id_hapus & id_kembali:
    raise ValueError(
        "Ada ID yang sekaligus ditandai untuk dihapus dan dipulihkan."
    )

# 6. Koreksi hasil filtering
relevant_final = relevant[
    ~relevant["id"].isin(id_hapus)
].copy()

dipulihkan = removed[
    removed["id"].isin(id_kembali)
].copy()

dipulihkan["filter_otomatis"] = "lolos"
dipulihkan["filter_reason"] = "manual_override_relevant"

relevant_final = pd.concat(
    [relevant_final, dipulihkan],
    ignore_index=True
)

# 7. Validasi dan simpan
assert relevant_final["id"].is_unique

relevant_final.to_csv(
    LABEL_DIR / "relevant_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Awal:", len(relevant))
print("Salah lolos dihapus:", len(id_hapus))
print("Salah dibuang dikembalikan:", len(id_kembali))
print("Dataset final:", len(relevant_final))


Awal: 8078
Salah lolos dihapus: 28
Salah dibuang dikembalikan: 22
Dataset final: 8072
